In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<b><font color="red" size="6">ch15. 데이터베이스 연동</font></b>
# 1절. SQLite 데이터 베이스 연결
- SQLite 데이터베이스는 별도의 DBMS없이 SQL을 이용해서 DB액세스할 수 있도록 만든 간단한 디스크기반 DB제공
- C라이브러리
- SQLite는 프로토타입을 만들 때 사용
- 프로젝트단계 : 분석  ->  설계  ->  구현  ->  테스트  ->  고객에게 배포  ->  유지보수
          -      프로토타입(SQLite) 시제품(구현후 반양산직전) 완제품(Oracle, MySQL, MariaDB, PostgreSQL, MSSQL, 아마존auroraDB, ...)

- [DB Browser for SQLite](https://sqlitebrowser.org/)에서 "DB Browser for SQLite - .zip (no installer) for 64-bit Windows" 다운로드후 압축 풀기

## 1.1 SQLite browser 설치 및 sqlite3패키지 load

In [6]:
import sqlite3
sqlite3.sqlite_version

'3.40.1'

In [5]:
import pandas as pd
pd.__version__

'1.5.3'

## 1.2 데이터베이스 연결
- 데이터베이스연결 객체 -> 커서객체(SQL전송 및 결과 받는 객체) -> 원하는 로직 수행 -> 커서객체 해제 -> DB연결객체 해제(close)
- SQLite로 DB 연결객체 생성시, DB파일이 있으면 연결, DB파일이 없으면 빈 DB파일 생성

In [7]:
# DB연결 (여기서 에러가 날 경우 VC_redist.x64.exe 설치)
conn = sqlite3.connect('data/ch15_example.db')
conn

In [8]:
# 커서 객체 생성 : 커서는 SQL문 실행시키고, 결과를 받는 객체
cursor = conn.cursor()
cursor

In [38]:
cursor.execute('''
    CREATE TABLE MEMBER (
        NAME TEXT,
        AGE  INT,
        EMAIL TEXT 
    )
''')

In [37]:
cursor.execute('DROP TABLE MEMBER')

In [41]:
sql = 'INSERT INTO MEMBER VALUES (\'마길동\', 25, \'h@h.com\')'
cursor.execute(sql) # sql 전송
print('insert, update, delete문의 수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신길동', 30, 's@h.com')"
cursor.execute(sql) 
print('수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신림동', 35, 'sil@h.com')"
cursor.execute(sql) 
print('수행 결과 행수 :', cursor.rowcount)

insert, update, delete문의 수행 결과 행수 : 1
수행 결과 행수 : 1
수행 결과 행수 : 1


In [26]:
print(cursor.rowcount)

1


In [42]:
conn.commit() # 反. conn.rollback()DML에서만 commit이나 rollback

In [45]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE") # SELECT sql문 전송

In [27]:
print(cursor.rowcount)

1


In [46]:
cursor.fetchmany(4)

[('마길동', 25, 'h@h.com'),
 ('마길동', 25, 'h@h.com'),
 ('마길동', 25, 'h@h.com'),
 ('신길동', 30, 's@h.com')]

In [29]:
# INSERT, UPDATE, DELETE문 실행결과 : cursor.rowcount
# SELECT문 실행결과를 받는 함수들
    # fetchone() : 결과를 한행씩 받을 때 (튜플)
    # fetchall() : 결과를 모두 받을 때 (튜플 list)
    # fetchmany(n) : 결과를 n행 받을 때(튜플 list)
cursor.fetchall()

[]

In [49]:
cursor.fetchall() # 한번 소요된 cursor 객체는 다시 fetch할 수 없음

[]

In [50]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchall()
members

[('마길동', 25, 'h@h.com'),
 ('마길동', 25, 'h@h.com'),
 ('마길동', 25, 'h@h.com'),
 ('신길동', 30, 's@h.com'),
 ('신길동', 30, 's@h.com'),
 ('신길동', 30, 's@h.com'),
 ('신림동', 35, 'sil@h.com'),
 ('신림동', 35, 'sil@h.com'),
 ('신림동', 35, 'sil@h.com')]

In [51]:
# 한줄씩 읽어서 dict list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    member = cursor.fetchone() # SQL문 수행결과를 한줄 가져오기
    if member is None:
        break
    members.append({'name':member[0], 'age':member[1], 'email':member[2]})
members

[{'name': '마길동', 'age': 25, 'email': 'h@h.com'},
 {'name': '마길동', 'age': 25, 'email': 'h@h.com'},
 {'name': '마길동', 'age': 25, 'email': 'h@h.com'},
 {'name': '신길동', 'age': 30, 'email': 's@h.com'},
 {'name': '신길동', 'age': 30, 'email': 's@h.com'},
 {'name': '신길동', 'age': 30, 'email': 's@h.com'},
 {'name': '신림동', 'age': 35, 'email': 'sil@h.com'},
 {'name': '신림동', 'age': 35, 'email': 'sil@h.com'},
 {'name': '신림동', 'age': 35, 'email': 'sil@h.com'}]

In [23]:
class Member:
    'Member 테이블의 내용을 받은 객체 타입'
    def __init__(self, name, age, email):
        self.name = name
        self.age  = age
        self.email = email
    def __str__(self):
        return "{}\t{}\t{}".format(self.name, self.age, self.email)
m = Member('홍길동', 25, 'h@h.com')
print(m)

홍길동	25	h@h.com


In [24]:
dbmember = ('홍길동', 25, 'h@h.com')
m = Member(*dbmember)
print(m)

홍길동	25	h@h.com


In [30]:
# 한줄씩 읽어서 객체list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    dbmember = cursor.fetchone()
    if dbmember is None:
        break
    member = Member(*dbmember)
    members.append(member)
for mem in members:
    print(mem)

홍길동	25	h@h.com
마길동	25	h@h.com
신길동	30	s@h.com
신림동	35	sil@h.com


In [31]:
# 최상위 n행 읽어오기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchmany(2)
members

[('홍길동', 25, 'h@h.com'), ('마길동', 25, 'h@h.com')]

In [33]:
cursor.close()
conn.close()

## 1.3 SQL구문에 파라미터 사용하기
- qmark(DB에 따라 불가한 경우가 있음)
- named(추천)

In [34]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN ('홍길동','신길동')")
cursor.fetchall()

[('홍길동', 25, 'h@h.com'), ('신길동', 30, 's@h.com')]

In [36]:
# 파라미터 사용하기 : qmark 방법 이용
name1 = input('검색할 이름1 :')
name2 = input('검색할 이름2 :')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (?, ?)", (name1, name2))
cursor.fetchall()

검색할 이름1 :신길동
검색할 이름2 :신림동


[('신길동', 30, 's@h.com'), ('신림동', 35, 'sil@h.com')]

In [37]:
# 파라미터 사용하기 : named 방법 이용
name1 = input('검색할 이름1 :')
name2 = input('검색할 이름2 :')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (:name1, :name2)", {'name1':name1,
                                                                      'name2':name2})
cursor.fetchall()

검색할 이름1 :마마마
검색할 이름2 :가가가


[]

In [39]:
# 파라미터 이용하기 : named 방법
name = input('회원가입할 이름은?')
try:
    age = int(input('나이는?(꼭 숫자로)'))
except:
    print('유효하지 않은 나이를 입력할 경우 1세로 초기화합니다')
    age = 1
email = input('이메일은?')
cursor.execute("INSERT INTO MEMBER VALUES (:name, :age, :email)", 
              {'name':name, 'age':age, 'email':email}) # sql 전송
conn.commit()
print('수행 결과 행수 :', cursor.rowcount)

회원가입할 이름은?박길동
나이는?(꼭 숫자로)이팔청춘
유효하지 않은 나이를 입력할 경우 1세로 초기화합니다
이메일은?l@l.com
수행 결과 행수 : 1


In [40]:
cursor.close()
conn.close()

# 2절. 오라클 데이터베이스 연결
- pip install cx_oracle(11g까지), pip install oracledb(12버전부터)

In [42]:
import cx_Oracle
cx_Oracle.__version__

'8.3.0'

In [45]:
# conn 얻어오는 방법1
oracle_dsn = cx_Oracle.makedsn(host="localhost", port=1521, sid='xe')
# conn = cx_Oracle.connect(user='scott', password='tiger', dsn=oracle_dsn)
conn = cx_Oracle.connect('scott', 'tiger', oracle_dsn)
conn.close()

In [49]:
# conn 얻어오는 방법 2 : (여기서 에러가 날 경우 VC_redist.x64.exe 설치)
#conn = cx_Oracle.connect(user='scott', password='tiger', dsn='localhost:1521/xe')
conn = cx_Oracle.connect('scott', 'tiger', 'localhost:1521/xe')
conn

<cx_Oracle.Connection to scott@localhost:1521/xe>

In [51]:
# cursor 객체 생성하고 sql문 전송&결과 받기
cursor = conn.cursor()
sql = "SELECT EMPNO NO, ENAME, JOB, MGR, HIREDATE, SAL, COMM, DEPTNO FROM EMP"
cursor.execute(sql)
emps = cursor.fetchall()

In [52]:
for emp in emps:
    print(emp)

(7369, 'SMITH', 'CLERK', 7902, datetime.datetime(1980, 12, 17, 0, 0), 800.0, None, 20)
(7499, 'ALLEN', 'SALESMAN', 7698, datetime.datetime(1981, 2, 20, 0, 0), 1600.0, 300.0, 30)
(7521, 'WARD', 'SALESMAN', 7698, datetime.datetime(1981, 2, 22, 0, 0), 1250.0, 500.0, 30)
(7566, 'JONES', 'MANAGER', 7839, datetime.datetime(1981, 4, 2, 0, 0), 2975.0, None, 20)
(7654, 'MARTIN', 'SALESMAN', 7698, datetime.datetime(1981, 9, 28, 0, 0), 1250.0, 1400.0, 30)
(7698, 'BLAKE', 'MANAGER', 7839, datetime.datetime(1981, 5, 1, 0, 0), 2850.0, None, 30)
(7782, 'CLARK', 'MANAGER', 7839, datetime.datetime(1981, 6, 9, 0, 0), 2450.0, None, 10)
(7788, 'SCOTT', 'ANALYST', 7566, datetime.datetime(1982, 12, 9, 0, 0), 3000.0, None, 20)
(7839, 'KING', 'PRESIDENT', None, datetime.datetime(1981, 11, 17, 0, 0), 5000.0, None, 10)
(7844, 'TURNER', 'SALESMAN', 7698, datetime.datetime(1981, 9, 8, 0, 0), 1500.0, 0.0, 30)
(7876, 'ADAMS', 'CLERK', 7788, datetime.datetime(1983, 1, 12, 0, 0), 1100.0, None, 20)
(7900, 'JAMES', 'CL

In [54]:
cursor.description

[('NO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 0),
 ('ENAME', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 10, 10, None, None, 1),
 ('JOB', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 9, 9, None, None, 1),
 ('MGR', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 1),
 ('HIREDATE', <cx_Oracle.DbType DB_TYPE_DATE>, 23, None, None, None, 1),
 ('SAL', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('COMM', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('DEPTNO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 3, None, 2, 0, 1)]

In [56]:
[descipt[0] for descipt in cursor.description]

['NO', 'ENAME', 'JOB', 'MGR', 'HIREDATE', 'SAL', 'COMM', 'DEPTNO']

In [57]:
import pandas as pd
emp_df = pd.DataFrame(emps, 
                     columns=[descipt[0] for descipt in cursor.description])
emp_df

,NO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,7902.0,1980-12-17,800.0,NaN,20
1,7499,ALLEN,SALESMAN,7698.0,1981-02-20,1600.0,300.0,30
2,7521,WARD,SALESMAN,7698.0,1981-02-22,1250.0,500.0,30
3,7566,JONES,MANAGER,7839.0,1981-04-02,2975.0,NaN,20
4,7654,MARTIN,SALESMAN,7698.0,1981-09-28,1250.0,1400.0,30
5,7698,BLAKE,MANAGER,7839.0,1981-05-01,2850.0,NaN,30
6,7782,CLARK,MANAGER,7839.0,1981-06-09,2450.0,NaN,10
7,7788,SCOTT,ANALYST,7566.0,1982-12-09,3000.0,NaN,20
8,7839,KING,PRESIDENT,NaN,1981-11-17,5000.0,NaN,10
9,7844,TURNER,SALESMAN,7698.0,1981-09-08,1500.0,0.0,30


In [65]:
# 사용자로부터 검색할 이름을 받아 해당 데이터 출력
sql = "SELECT * FROM EMP WHERE ENAME=(:ename)"
ename = input('검색할 이름?').upper()
cursor.execute(sql, {'ename':ename})
emp = cursor.fetchone() # 결과가 있으면 해당 데이터를 튜플로, 결과가 없으면 None으로
if emp:
    columns = [descript[0] for descript in cursor.description]
    df = pd.DataFrame([emp], columns=columns)
    display(df)
else:
    print('해당 이름이 없습니다')

검색할 이름?scott


,EMPNO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7788,SCOTT,ANALYST,7566,1982-12-09,3000.0,None,20


In [68]:
for col, data in zip(columns, emp):
    print("{}:{}".format(col, data if data is not None else '-'))

EMPNO:7788
ENAME:SCOTT
JOB:ANALYST
MGR:7566
HIREDATE:1982-12-09 00:00:00
SAL:3000.0
COMM:-
DEPTNO:20


In [69]:
cursor.close()
conn.close()

# 3절. MySQL 연결

| 라이브러리 | 특징 |
|:--|:--|
| **mysql-connector-python** | MySQL 공식 커넥터. python으로 구현 |
| **pyMySQL** | 커뮤니티에서 만든 서드파트 라이브러리(경량). python으로 구현. 널리 쓰임 |

- pip install pyMySQL

In [52]:
%pip install pyMySQL

     ---------------------------------------- 45.7/45.7 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [64]:
import pymysql
conn = pymysql.connect(
    host='127.0.0.1', #또는 'localhost'
    #port=3306,  굳이 안써도 됨 알아서 찾아씀
    user='root',
    password='mysql',
    database='devdb',
    charset='utf8mb4', #이모지나 한글 깨짐 방지
    autocommit=True    # 자동 커밋 
)
cursor= conn.cursor()
sql='select * from person'
cursor.execute(sql) #sql전송하고 결과 받기
person=cursor.fetchall()
cursor.close()
conn.close()

In [71]:
df=pd.DataFrame(person)
df

,0,1,2,3,4,5,6,7
0,1001,bill,president,NaN,1989-01-10,7000,None,10
1,1111,smith,manager,1001.0,1990-12-17,1000,None,10
2,1112,ally,salesman,1116.0,1991-02-20,1600,500,30
3,1113,word,salesman,1116.0,1992-02-24,1450,300,30
4,1114,james,manager,1001.0,1990-04-12,3975,None,20
5,1116,johnson,manager,1001.0,1991-05-01,3550,None,30
6,1118,martin,analyst,1111.0,1991-09-09,3450,None,10
7,1121,kim,clerk,1114.0,1990-12-08,4000,None,20
8,1123,lee,salesman,1116.0,1991-09-23,1200,0,30
9,1226,park,analyst,1111.0,1990-01-03,2500,None,10


In [66]:
import pandas as pd
import numpy as np

In [68]:
[descript[0] for descript in cursor.description]

['pno', 'pname', 'job', 'manager', 'hiredate', 'sal', 'comm', 'dno']

In [73]:
df=pd.DataFrame(person,columns=[descript[0] for descript in cursor.description])

In [83]:
df.isna().sum()

pno         0
pname       0
job         0
manager     1
hiredate    0
sal         0
comm        7
dno         0
dtype: int64

In [85]:
df['hiredate']=df['hiredate'].astype('datetime64[ns]')

In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   pno       10 non-null     int64         
 1   pname     10 non-null     object        
 2   job       10 non-null     object        
 3   manager   9 non-null      float64       
 4   hiredate  10 non-null     datetime64[ns]
 5   sal       10 non-null     object        
 6   comm      3 non-null      object        
 7   dno       10 non-null     int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 768.0+ bytes


In [87]:
df=df.astype({'sal':'float64', 'comm':np.float64})

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   pno       10 non-null     int64         
 1   pname     10 non-null     object        
 2   job       10 non-null     object        
 3   manager   9 non-null      float64       
 4   hiredate  10 non-null     datetime64[ns]
 5   sal       10 non-null     float64       
 6   comm      3 non-null      float64       
 7   dno       10 non-null     int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(2)
memory usage: 768.0+ bytes


# 4절. 연습문제
## oracle연동 
- 회원가입 | 전체조회 | 이름찾기 | 메일삭제 | CSV내보내기 | 종료
### 0. 처음실행

In [1]:
def load_conn():
    global conn # 변수를 전역변수로 쓰겠다
    import cx_Oracle
    conn = cx_Oracle.connect("scott","tiger", "localhost:1521/xe")
load_conn()

In [97]:
!pip install cx_Oracle

     ------------------------------------- 213.1/213.1 kB 13.5 MB/s eta 0:00:00


### 1. 회원입력

In [2]:
def fn1_insert_member():
    '사용자로부터 이름, 전화, 이메일, 나이, 등급(1~5)을 입력받아 DB에 insert한다'
    # 사용자로부터 데이터 받기
    name = input('이름: ')
    phone= input('전화: ')
    email = input('메일: ')
    try:
        age = int(input('나이: '))
        if (age<0) or (age>150):
            age = 0
    except:
        print('유효하지 않은 나이 입력시 나이는 0으로 최기화')
        age=0
    try:
        grade = int(input('등급(1~5): '))
        if grade<1:
            grade=1
        elif grade > 5:
            grade=5
    except:
        print('유효하지 않은 등급 입력시 등급은 1으로 최기화')
        grade=1    
    # SQL 전송 및 결과 받기
    cursor=conn.cursor()
    sql = "INSERT INTO MEMBER VALUES (:name, :phone, :email, :age, :grade)"
    cursor.execute(sql, 
                   {'name':name,
                    'phone':phone, 
                    'email':email,
                    'age':age, 
                    'grade':grade}) # sql전송 및 결과 받기
    if cursor.rowcount:
        conn.commit()
        print(name+ '님 회원가입 완료')
    cursor.close()


### 2. 전체조회

In [3]:
def fn2_display_members():
    'member 테이블의 내용을 데이터프레임으로 display'
    import pandas as pd 
    cursor = conn.cursor()
    sql="""SELECT * 
            FROM MEMBER 
            ORDER BY AGE"""
    cursor.execute(sql)
    members=cursor.fetchall() # tuple list (데이터가 없으면 빈list)
    if members:
        columns =[descript[0] for descript in cursor.description]
        df = pd.DataFrame(members,columns=columns)
        display(df)
    else:
        print('입력된 회원이 없습니다.')
    cursor.close()
fn2_display_members()

,NAME,PHONE,EMAIL,AGE,GRADE
0,23,23,23,23,5
1,홍길동,010-9999-9999,HONG@NAVER,25,1
2,홍길동,010-9999-9999,HONG@NAVER,25,1
3,홍길동,010-9999-9999,HONG@NAVER,25,1


### 3. 이름으로 조회하기


In [4]:
def fn3_search_name():
    import pandas as pd 
    search_name = input('조회할 이름: ')
    cursor = conn.cursor()
    sql = "SELECT * FROM MEMBER WHERE NAME = :search_name"
    cursor.execute(sql,{"search_name": search_name})
    search_member=cursor.fetchall()
    if search_member:
        columns =[descript[0] for descript in cursor.description]
        df = pd.DataFrame(search_member,columns=columns)
        display(df)
    else:
        print('찾으시는 회원이름은 없습니다.')
    cursor.close()
fn3_search_name() 

조회할 이름: 홍길동


,NAME,PHONE,EMAIL,AGE,GRADE
0,홍길동,010-9999-9999,HONG@NAVER,25,1
1,홍길동,010-9999-9999,HONG@NAVER,25,1
2,홍길동,010-9999-9999,HONG@NAVER,25,1


### 4. 메일로삭제

In [5]:
def fn4_delete_member():
    '삭제하고자 하는 회원의 메일을 입력받아 회원정보 delete하기'
    delete_member = input('삭제할 사람의 이메일: ')
    cursor = conn.cursor()
    sql = "SELECT NAME FROM MEMBER WHERE UPPER(EMAIL)=UPPER(:email)"
    cursor.execute(sql,{"email": delete_member}) 
    members=cursor.fetchall() # [(홍길동,)]
    if members:
        names = [member[0] for member in members]
        sql = "DELETE FROM MEMBER WHERE EMAIL = :delete_member"
        cursor.execute(sql,{"delete_member": delete_member})
        conn.commit()
        print(f"{names}'님 데이터가 삭제되었습니다")
    else:
        print('입력하신 이메일의 회원이 없습니다')
    cursor.close()

### 5. CSV 내보내기
- data/ch15_member.csv로 회원정보 내보내기

In [6]:
def fn5_save_csv():
    import pandas as pd 
    cursor = conn.cursor()
    sql="""SELECT * 
            FROM MEMBER 
            ORDER BY AGE"""
    cursor.execute(sql)
    members=cursor.fetchall() # tuple list (데이터가 없으면 빈list)
    columns =[descript[0] for descript in cursor.description]
    df = pd.DataFrame(members,columns=columns)
    display(df)
    cursor.close()
    df.to_csv('data/ch15_member.csv',index=False)
    print('csv파일 백업완료')
fn5_save_csv()

,NAME,PHONE,EMAIL,AGE,GRADE
0,23,23,23,23,5
1,홍길동,010-9999-9999,HONG@NAVER,25,1
2,홍길동,010-9999-9999,HONG@NAVER,25,1
3,홍길동,010-9999-9999,HONG@NAVER,25,1


csv파일 백업완료


### 6. 기능합치기

In [ ]:
def main():
    while True:
        menu = input("1:입력 | 2:전체조회 | 3:이름찾기 | 4:메일삭제 | 5:CSV백업 | 6:종료")
        if menu=="1":
            fn1_insert_member()
        elif menu=="2":
            fn2_display_members()
        elif menu=="3":
            fn3_search_name()        
        elif menu=="4":
            fn4_delete_member()
        elif menu=="5":
            fn5_save_csv() 
        elif menu=="6":
             conn.close()
             break
        else:
            print('유효한 메뉴번호를 입력해주세요')
if __name__=='__main__':
    import cx_Oracle
    conn = cx_Oracle.connect("scott","tiger","localhost:1521/xe")
    main()
    
    #sqlite3에서는 
#     import sqlite3
#     conn= sqlite3.connect('data/ch15_member.db')
#     main()
    
    

1:입력 | 2:전체조회 | 3:이름찾기 | 4:메일삭제 | 5:CSV백업 | 6:종료2


,NAME,PHONE,EMAIL,AGE,GRADE
0,23,23,23,23,5
1,홍길동,010-9999-9999,HONG@NAVER,25,1
2,홍길동,010-9999-9999,HONG@NAVER,25,1
3,홍길동,010-9999-9999,HONG@NAVER,25,1


1:입력 | 2:전체조회 | 3:이름찾기 | 4:메일삭제 | 5:CSV백업 | 6:종료3
조회할 이름: 길동
찾으시는 회원이름은 없습니다.
1:입력 | 2:전체조회 | 3:이름찾기 | 4:메일삭제 | 5:CSV백업 | 6:종료3
조회할 이름: 홍길동


,NAME,PHONE,EMAIL,AGE,GRADE
0,홍길동,010-9999-9999,HONG@NAVER,25,1
1,홍길동,010-9999-9999,HONG@NAVER,25,1
2,홍길동,010-9999-9999,HONG@NAVER,25,1
